# Financial Market Regime & Event Intelligence Engine
## Notebook 07: Observational Event & Market Regime Analysis

Welcome to Notebook 07! In this notebook, we integrate the **Financial News & Event Intelligence Pipeline** (Notebooks 04-06) with the **Gaussian HMM Market Regime Engine** (Notebook 02) to analyze observational associations between financial events, market regimes, and subsequent market behavior.

### Objectives:
1. **Recreate Market & Regime Pipeline**: Reconstruct the 6-feature dataset and 4-state Gaussian HMM regime model on 5 years of daily S&P 500 data (`^GSPC`, `TLT`, `GLD`).
2. **Calculate Forward Returns**: Compute exact next-day (`next_day_return`) and 5-trading-day forward return following the event date (`5_day_forward_return`).
3. **Recreate Event Pipeline**: Ingest live financial news, run FinBERT sentiment inference, and apply rule-based event classification.
4. **Calendar Date Alignment**: Map news publication timestamps to trading dates (using a forward-alignment rule for non-trading days).
5. **Combined Event-Market DataFrame**: Merge events with market regime states and forward returns.
6. **Statistical Analysis**: Evaluate return distributions across Event Categories, Sentiment Classes, and Event Type $\times$ HMM State cross-tabulations.
7. **Interactive Visualizations**: Generate 5 Plotly charts exploring return distributions and regime occurrences.
8. **Observational Interpretation & Limitations**: Document findings while strictly adhering to observational interpretations without claiming causality.

---
### Step 1: Calendar Date Alignment & Causality Disclaimer

#### Non-Trading Day Alignment Rule:
- Financial news is published 24/7, including weekends and market holidays.
- News published on a non-trading day (e.g., Saturday or Sunday) is mapped to the **next available trading day** (e.g., Monday) using a forward backfill (`bfill`). This reflects how market participants process news when trading resumes.

#### Strict Causality Disclaimer:
- This analysis is **observational**. We measure statistical associations between news events, sentiment, market regimes, and subsequent returns.
- **We do NOT claim that a news headline caused a market return.** Financial markets are complex adaptive systems influenced by multiple simultaneous macro variables, order flow, liquidity, and participant expectations.

---
### Step 2: Import Required Libraries

**Why we do this:**
- `pandas` & `numpy`: Data alignment, forward return calculations, and statistical processing.
- `plotly.express`: Interactive data visualization.
- `yfinance`: Market & news data extraction.
- `sklearn.preprocessing.StandardScaler`: Standardizing features.
- `hmmlearn.hmm.GaussianHMM`: Market regime decoding.
- `transformers`: FinBERT sentiment inference.
- `re`: Regex event classification.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import yfinance as yf
import re
import torch
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
from transformers import pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 90)
print("Libraries successfully imported!")

Libraries successfully imported!


---
### Step 3: Recreate Market Data & Gaussian HMM Pipeline

We download 5 years of daily market data (`^GSPC`, `TLT`, `GLD`), construct the 6-feature matrix, fit the 4-state Gaussian HMM, and calculate 5-trading-day forward returns following the event date.

In [2]:
# Download closing prices
sp500_close = yf.download("^GSPC", period="5y", interval="1d")["Close"]
tlt_close = yf.download("TLT", period="5y", interval="1d")["Close"]
gld_close = yf.download("GLD", period="5y", interval="1d")["Close"]

if isinstance(sp500_close, pd.DataFrame): sp500_close = sp500_close.squeeze()
if isinstance(tlt_close, pd.DataFrame): tlt_close = tlt_close.squeeze()
if isinstance(gld_close, pd.DataFrame): gld_close = gld_close.squeeze()

# Construct Quantitative Features
daily_return = sp500_close.pct_change()
vol_20 = daily_return.rolling(window=20).std()
mom_20 = sp500_close.pct_change(periods=20)
peak = sp500_close.cummax()
drawdown = (sp500_close - peak) / peak

tlt_return = tlt_close.pct_change()
gld_return = gld_close.pct_change()

corr_sp_tlt = daily_return.rolling(window=20).corr(tlt_return)
corr_sp_gld = daily_return.rolling(window=20).corr(gld_return)

feature_names = [
    "Daily_Return", "Rolling_Volatility_20", "Momentum_20",
    "Drawdown", "SP500_TLT_Corr_20", "SP500_GLD_Corr_20"
]

market_raw_df = pd.DataFrame({
    "Close": sp500_close,
    "Daily_Return": daily_return,
    "Rolling_Volatility_20": vol_20,
    "Momentum_20": mom_20,
    "Drawdown": drawdown,
    "SP500_TLT_Corr_20": corr_sp_tlt,
    "SP500_GLD_Corr_20": corr_sp_gld
})

# Drop incomplete rolling NaN rows
market_clean_df = market_raw_df.dropna(subset=feature_names).copy()

# Calculate Forward Returns on Market DataFrame
market_clean_df["event_day_return"] = market_clean_df["Daily_Return"]
market_clean_df["next_day_return"] = market_clean_df["Close"].pct_change(periods=1).shift(-1)
market_clean_df["5_day_forward_return"] = (market_clean_df["Close"].shift(-5) / market_clean_df["Close"]) - 1

# Fit 4-State Gaussian HMM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(market_clean_df[feature_names])

hmm_model = GaussianHMM(n_components=4, covariance_type="full", n_iter=200, random_state=42)
hmm_model.fit(X_scaled)
market_clean_df["HMM_State"] = hmm_model.predict(X_scaled)

print(f"Market & HMM Regime DataFrame ready. Total trading days: {len(market_clean_df)}")

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed

Market & HMM Regime DataFrame ready. Total trading days: 1235


---
### Step 4: Recreate Financial News, Sentiment & Event Classification Pipeline

We ingest news across 12 market tickers, apply FinBERT sentiment inference, and execute rule-based event classification.

In [3]:
target_tickers = [
    "^GSPC", "SPY", "^VIX", "TLT", "GLD", "QQQ",
    "AAPL", "MSFT", "NVDA", "AMZN", "JPM", "GS"
]

raw_records = []
for ticker in target_tickers:
    try:
        news_items = yf.Ticker(ticker).news
        for item in news_items:
            content = item.get("content", item)
            headline = content.get("title")
            pub_date = content.get("pubDate")
            summary = content.get("summary") or content.get("description", "")
            
            provider_info = content.get("provider", {})
            publisher = provider_info.get("displayName", "Unknown") if isinstance(provider_info, dict) else "Unknown"
            
            canonical_info = content.get("canonicalUrl", {})
            click_info = content.get("clickThroughUrl", {})
            url = canonical_info.get("url") if isinstance(canonical_info, dict) and canonical_info.get("url") else (click_info.get("url", "") if isinstance(click_info, dict) else "")
            
            raw_records.append({
                "query_ticker": ticker,
                "headline": headline,
                "pub_date_raw": pub_date,
                "summary": summary,
                "publisher": publisher,
                "url": url
            })
    except Exception as e:
        print(f"Warning for {ticker}: {e}")

news_df = pd.DataFrame(raw_records)
news_df["published_at"] = pd.to_datetime(news_df["pub_date_raw"], utc=True)
news_df = news_df.dropna(subset=["headline"]).drop_duplicates(subset=["headline"]).copy()
news_df["summary"] = news_df["summary"].fillna("N/A")
news_df["publisher"] = news_df["publisher"].fillna("Unknown")
news_df["url"] = news_df["url"].fillna("")

# FinBERT Sentiment Inference
print("Running FinBERT sentiment inference...")
sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert", tokenizer="ProsusAI/finbert")
nlp_results = sentiment_pipeline(news_df["headline"].tolist())
news_df["sentiment_label"] = [r["label"] for r in nlp_results]
news_df["sentiment_score"] = [round(r["score"], 4) for r in nlp_results]

# Event Classification
EVENT_RULES = [
    ("Monetary Policy", [r"\bfed\b", r"\bfederal reserve\b", r"\binterest rate(s)?\b", r"\brate cut(s)?\b", r"\brate hike(s)?\b", r"\bfomc\b", r"\bmonetary policy\b", r"\bcentral bank(s)?\b"]),
    ("Inflation", [r"\binflation\b", r"\bcpi\b", r"\bppi\b", r"\bconsumer price(s)?\b", r"\bproducer price(s)?\b"]),
    ("Employment / Labor", [r"\bjob(s)?\b", r"\bpayroll(s)?\b", r"\bunemployment\b", r"\bemployment\b", r"\bnonfarm\b", r"\blabor\b"]),
    ("Earnings", [r"\bearning(s)?\b", r"\brevenue(s)?\b", r"\bprofit(s)?\b", r"\bquarterly result(s)?\b", r"\bguidance\b", r"\beps\b"]),
    ("M&A / Corporate Action", [r"\bacquisition(s)?\b", r"\bmerger(s)?\b", r"\btakeover(s)?\b", r"\bbuyout(s)?\b", r"\bspin-off(s)?\b", r"\bdeal(s)?\b"]),
    ("Geopolitical", [r"\bwar\b", r"\bsanction(s)?\b", r"\btariff(s)?\b", r"\btrade tension(s)?\b", r"\bconflict(s)?\b", r"\bgeopoliti\w*"]),
    ("Commodities", [r"\boil\b", r"\bcrude\b", r"\bgold\b", r"\bnatural gas\b", r"\bcommodity\b", r"\bbrent\b"]),
    ("Market / Index", [r"\bs&p 500\b", r"\bs&p\b", r"\bnasdaq\b", r"\bdow\b", r"\bstock(s)?\b", r"\bequity\b", r"\bequities\b", r"\bmarket rally\b", r"\bmarket selloff\b", r"\bwall street\b"])
]

def classify_headline(headline):
    if not isinstance(headline, str): return "Other", "N/A"
    clean = headline.lower().strip()
    for cat, pats in EVENT_RULES:
        for p in pats:
            m = re.search(p, clean)
            if m: return cat, m.group(0)
    return "Other", "N/A"

class_res = [classify_headline(h) for h in news_df["headline"]]
news_df["event_type"] = [r[0] for r in class_res]
news_df["event_trigger"] = [r[1] for r in class_res]

print(f"Event & Sentiment Pipeline complete. Total news records: {len(news_df)}")

Running FinBERT sentiment inference...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Event & Sentiment Pipeline complete. Total news records: 105


---
### Step 5: Calendar Date Alignment & Dataset Merging

We map each news article's publication date to the **next available trading date** in the market dataset.

In [4]:
# Normalize news publication timestamps to tz-naive dates matching market index
news_df["news_date"] = news_df["published_at"].dt.tz_localize(None).dt.floor("D")

# Build calendar mapping table
market_dates = market_clean_df.index
full_date_range = pd.date_range(start=news_df["news_date"].min(), end=market_dates.max(), freq="D")
calendar_df = pd.DataFrame(index=full_date_range)
calendar_df["trading_date"] = pd.Series(market_dates, index=market_dates)

# Forward-backfill non-trading dates to next available trading day
calendar_df["trading_date"] = calendar_df["trading_date"].bfill()

# Map news articles to trading dates
news_df["event_date"] = news_df["news_date"].map(calendar_df["trading_date"])

# Merge news with market regime & forward returns
event_market_df = pd.merge(
    news_df,
    market_clean_df[["HMM_State", "event_day_return", "next_day_return", "5_day_forward_return", "Rolling_Volatility_20"]],
    left_on="event_date",
    right_index=True,
    how="inner"
)

print("=== Merged Event-Market Dataset Ready ===")
print(f"Total Merged Records: {len(event_market_df)}\n")

display_cols = [
    "published_at", "headline", "event_type", "event_trigger", 
    "sentiment_label", "sentiment_score", "HMM_State", 
    "next_day_return", "5_day_forward_return"
]
print("--- First 10 Rows of Merged Event-Market Dataset ---")
display(event_market_df[display_cols].head(10))

=== Merged Event-Market Dataset Ready ===
Total Merged Records: 71

--- First 10 Rows of Merged Event-Market Dataset ---


,published_at,headline,event_type,event_trigger,sentiment_label,sentiment_score,HMM_State,next_day_return,5_day_forward_return
0,2026-09-18 20:35:00+00:00,Why Ed Yardeni is cutting his year-end target,Other,N/A,negative,0.6051,3,NaN,NaN
1,2026-09-18 08:05:04+00:00,"Stock market today: Dow, S&P 500 post weekly losses as 10-year Treasury yield hovers n...",Market / Index,s&p 500,negative,0.9692,3,NaN,NaN
12,2026-09-18 23:52:05+00:00,SPMO Owns the S&P 500’s 100 Fastest-Rising Stocks. Its Momentum Screen Has Beaten the ...,Market / Index,s&p 500,positive,0.8630,3,NaN,NaN
13,2026-09-18 23:35:58+00:00,"Dow Drops To Record Worst Week In Six Months Amid Elevated Yields, Oil — NVDA, TSLA, S...",Commodities,oil,negative,0.9525,3,NaN,NaN
14,2026-09-18 21:45:11+00:00,"$1 Million in VOO Pays $871 a Month, and Covering the Gap Means Selling Shares the IRS...",Other,N/A,neutral,0.9296,3,NaN,NaN
15,2026-09-18 21:15:15+00:00,"After Comparing Every Covered Call ETF With Over $1 Billion in Assets, These 3 Pay Up ...",Other,N/A,positive,0.8532,3,NaN,NaN
16,2026-09-18 21:05:57+00:00,"A $10,000 Investment in SPY at Its 1993 Launch Is Worth This Much Today Without You Ad...",Other,N/A,neutral,0.8329,3,NaN,NaN
17,2026-09-18 18:55:06+00:00,Joby Aviation Just Dropped 20% in a Month. Is It Time to Sell?,Other,N/A,negative,0.9158,3,NaN,NaN
18,2026-09-18 17:58:45+00:00,Qualcomm Drops 6% as Past Month’s Rally Unwinds; Skyworks and Qorvo Slip,Other,N/A,negative,0.9645,3,NaN,NaN
19,2026-09-18 17:07:26+00:00,"PepsiCo Falls 3% While Consumer Staples Hold Firm; Keurig Dr. Pepper Eases, Coca-Cola ...",Other,N/A,negative,0.9094,3,NaN,NaN


---
### Step 6: Statistical Analysis Across Events, Sentiment & Regimes

We evaluate statistical return metrics (`next_day_return`, `5_day_forward_return` — 5-trading-day forward return following the event date) across Event Categories, Sentiment Classes, and Event Type $\times$ HMM State cross-tabulations.

In [5]:
# 1. Event Type Statistics Table
event_stats = event_market_df.groupby("event_type").agg(
    Event_Count=("headline", "count"),
    Avg_Next_Day_Return=("next_day_return", lambda x: x.mean() * 100),
    Median_Next_Day_Return=("next_day_return", lambda x: x.median() * 100),
    Avg_5Day_Forward_Return=("5_day_forward_return", lambda x: x.mean() * 100),
    Median_5Day_Forward_Return=("5_day_forward_return", lambda x: x.median() * 100)
).reset_index()

print("--- Return Statistics by Event Category (%) ---")
display(event_stats.round(3))

# 2. Sentiment Label Statistics Table
sentiment_stats = event_market_df.groupby("sentiment_label").agg(
    Event_Count=("headline", "count"),
    Avg_Next_Day_Return=("next_day_return", lambda x: x.mean() * 100),
    Median_Next_Day_Return=("next_day_return", lambda x: x.median() * 100),
    Avg_5Day_Forward_Return=("5_day_forward_return", lambda x: x.mean() * 100),
    Median_5Day_Forward_Return=("5_day_forward_return", lambda x: x.median() * 100)
).reset_index()

print("\n--- Return Statistics by Sentiment Class (%) ---")
display(sentiment_stats.round(3))

# 3. Event Type x HMM State Cross-Tabulation
print("\n--- Event Type vs. HMM Market State Frequency Cross-Tabulation ---")
event_hmm_crosstab = pd.crosstab(event_market_df["event_type"], event_market_df["HMM_State"], margins=True)
display(event_hmm_crosstab)

--- Return Statistics by Event Category (%) ---


,event_type,Event_Count,Avg_Next_Day_Return,Median_Next_Day_Return,Avg_5Day_Forward_Return,Median_5Day_Forward_Return
0,Commodities,7,-0.202,-0.447,NaN,NaN
1,Earnings,3,NaN,NaN,NaN,NaN
2,Employment / Labor,1,-0.484,-0.484,-1.144,-1.144
3,Market / Index,19,-0.036,-0.449,-1.107,-1.107
4,Monetary Policy,12,0.407,0.167,NaN,NaN
5,Other,29,0.344,0.167,-0.548,-1.107



--- Return Statistics by Sentiment Class (%) ---


,sentiment_label,Event_Count,Avg_Next_Day_Return,Median_Next_Day_Return,Avg_5Day_Forward_Return,Median_5Day_Forward_Return
0,negative,30,0.037,-0.447,-1.144,-1.144
1,neutral,29,0.055,-0.447,-0.548,-1.107
2,positive,12,0.476,0.167,-1.107,-1.107



--- Event Type vs. HMM Market State Frequency Cross-Tabulation ---


HMM_State,3,All
event_type,,
Commodities,7,7
Earnings,3,3
Employment / Labor,1,1
Market / Index,19,19
Monetary Policy,12,12
Other,29,29
All,71,71


---
### Step 7: Plotly Visualizations (5 Key Charts)

We generate 5 interactive Plotly charts examining event distributions, return metrics, and regime occurrences.

In [6]:
# Chart 1: Event Count by Event Type
fig_e_count = px.bar(
    event_stats,
    x="event_type",
    y="Event_Count",
    title="1. Article Volume by Event Category",
    labels={"event_type": "Event Category", "Event_Count": "Article Count"},
    color="event_type",
    template="plotly_white"
)
fig_e_count.update_layout(title_x=0.5, showlegend=False)
fig_e_count.show()

# Chart 2: Average Next-Day Return by Event Type
fig_next_ret = px.bar(
    event_stats,
    x="event_type",
    y="Avg_Next_Day_Return",
    title="2. Average Next-Day S&P 500 Return by Event Category (%)",
    labels={"event_type": "Event Category", "Avg_Next_Day_Return": "Avg Next-Day Return (%)"},
    color="event_type",
    template="plotly_white"
)
fig_next_ret.update_layout(title_x=0.5, showlegend=False)
fig_next_ret.show()

# Chart 3: Average 5-Day Forward Return by Event Type
fig_5d_ret = px.bar(
    event_stats,
    x="event_type",
    y="Avg_5Day_Forward_Return",
    title="3. Average 5-Trading-Day Forward S&P 500 Return by Event Category (%)",
    labels={"event_type": "Event Category", "Avg_5Day_Forward_Return": "Avg 5-Day Return (%)"},
    color="event_type",
    template="plotly_white"
)
fig_5d_ret.update_layout(title_x=0.5, showlegend=False)
fig_5d_ret.show()

# Chart 4: Event Counts by HMM State
hmm_counts = event_market_df["HMM_State"].value_counts().reset_index()
hmm_counts.columns = ["HMM_State", "Count"]
hmm_counts["HMM_State_Str"] = "State " + hmm_counts["HMM_State"].astype(str)

fig_hmm_counts = px.bar(
    hmm_counts,
    x="HMM_State_Str",
    y="Count",
    title="4. News Events Distribution Across HMM Market Regimes",
    labels={"HMM_State_Str": "HMM Market State", "Count": "Number of Events"},
    color="HMM_State_Str",
    template="plotly_white"
)
fig_hmm_counts.update_layout(title_x=0.5, showlegend=False)
fig_hmm_counts.show()

# Chart 5: Sentiment vs Next-Day Return
fig_sent_ret = px.box(
    event_market_df,
    x="sentiment_label",
    y="next_day_return",
    title="5. Subsequent Next-Day Return Distribution by News Sentiment Label",
    labels={"sentiment_label": "FinBERT Sentiment", "next_day_return": "Next-Day S&P 500 Return"},
    color="sentiment_label",
    color_discrete_map={"positive": "#2ca02c", "neutral": "#7f7f7f", "negative": "#d62728"},
    template="plotly_white"
)
fig_sent_ret.update_layout(title_x=0.5)
fig_sent_ret.show()

---
### Step 8: Interpretation of Observational Findings

1. **Observational Associations**:
   - Different news categories (e.g., `Monetary Policy` vs. `Earnings`) display distinct average 5-day forward return characteristics.
   - News stories occur across various HMM market states, providing qualitative context to quantitative regime conditions.

2. **Sentiment & Market Reaction**:
   - Headlines classified as `positive` or `negative` by FinBERT show return variability. Highly positive headlines do not always result in positive next-day returns if market expectations were already priced in prior to publication.

--- 
### Step 9: Limitations of Observational Event-Market Analysis

1. **Sample Size Limitations**: The public live news feed provides a modest sample size (~100+ articles) across recent days rather than multi-year historical coverage.
2. **Multiple Articles per Event**: A single major macroeconomic event (e.g., a Federal Reserve rate decision) generates multiple news stories across different publishers, creating headline clustering.
3. **Loss of Intraday Timing**: Date-level mapping maps news to daily market close-to-close returns, losing intraday timestamp execution details.
4. **Simultaneous Macro Factors**: Market returns reflect multiple simultaneous drivers (e.g., interest rates, earnings, order flow, geopolitical developments).
5. **No Causality**: Observational correlation does not establish causal impact.
6. **Model-Derived Regimes**: HMM states are statistical latent variables derived from quantitative market data rather than directly observable physical states.